# All Methods + Backtest — BTC **Daily**

Same full comparison as the 4-hourly notebook, but on **daily** data. Daily returns are less noisy than 4h (less microstructure noise, stronger trend), so this is the best shot at finding a real predictive signal.

**13 variants × 2 target modes (3 seeds each) = 4 experiments**, then an automatic **portfolio backtest of every return-mode variant** with cross-variant comparison figures.

## Experiments

| # | Family | Mode | Base config |
|---|---|---|---|
| 1 | OHLC (7 var) | price | `daily/btc_ohlc.yaml` |
| 2 | OHLC (7 var) | return | `daily/btc_ohlc_return.yaml` |
| 3 | Hier (6 var) | price | `daily/btc_hier.yaml` |
| 4 | Hier (6 var) | return | `daily/btc_hier_return.yaml` |

Daily uses all ~2,191 bars (no `last_n_days` filter), window 20 days, 70/10/20 split.

## What you get
- Per-experiment comparison figures (metric bars, box plots, radar)
- Aggregate metrics table + signal-quality table (corr, dir-agree)
- For every return-mode experiment: backtest of all variants → comparison bars, signal-quality scatter grid, equity overlay vs buy & hold

## Runtime
Lighter than 4h (fewer bars). 4 experiments × 13 variants × 3 seeds on a T4 — a couple of hours. Run unattended.

**Requires:** `lunarcrush_btc_day_full.csv` in Google Drive.

## 1. Setup

In [ ]:
import os
if not os.path.exists('/content/thesis'):
    !git clone https://github.com/BerkayClik/thesis.git /content/thesis
%cd /content/thesis
!git pull

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
# Copy the BTC daily cache CSV from Drive (recursive search).
import os, shutil, glob
os.makedirs('data/cache', exist_ok=True)
DRIVE_DATA_DIR = '/content/drive/MyDrive/thesis_data'   # <-- the folder you uploaded to
copied = 0
if os.path.isdir(DRIVE_DATA_DIR):
    for src in glob.glob(os.path.join(DRIVE_DATA_DIR, '**/lunarcrush_btc_day*.csv'), recursive=True):
        shutil.copy(src, os.path.join('data/cache', os.path.basename(src))); copied += 1
        print('copied', os.path.basename(src))
if copied == 0:
    print(f'No BTC daily CSV found under {DRIVE_DATA_DIR} (searched recursively).')
!ls -la data/cache/lunarcrush_btc_day*.csv 2>/dev/null || echo 'missing BTC daily cache'

In [ ]:
!pip install -q yfinance scipy seaborn uv
import torch
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
os.environ['PYTHONPATH'] = '/content/thesis'
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

In [ ]:
# Build the isolated vectorbt backtest env once (needs numpy<2)
!uv venv --python 3.11 .venv-backtest
!uv pip install --python .venv-backtest -r requirements-backtest.txt
!.venv-backtest/bin/python scripts/backtest_env_smoke.py

## 2. Run the 4 experiments (13 variants × 3 seeds each)

In [ ]:
EXP = [
    ('btc_ohlc',        'daily_comparison_3seed'),     # 1 OHLC price
    ('btc_ohlc_return', 'daily_comparison_3seed'),     # 2 OHLC return
    ('btc_hier',        'daily_hierarchical_3seed'),   # 3 Hier price
    ('btc_hier_return', 'daily_hierarchical_3seed'),   # 4 Hier return
]
for i, (base, exp) in enumerate(EXP, 1):
    print(f'\n{"="*70}\nEXPERIMENT {i}/4: {base}  /  {exp}\n{"="*70}')
    !python experiments/run_experiments.py \
        --base-config configs/data/daily/{base}.yaml \
        --experiment-config configs/experiments/{exp}.yaml

## 3. Aggregate tables

In [ ]:
import json, glob, os
import numpy as np, pandas as pd

RESULT_DIRS = {
    'OHLC·price':  'experiments/results/daily_btc_ohlc',
    'OHLC·return': 'experiments/results/daily_btc_ohlc_return',
    'Hier·price':  'experiments/results/daily_btc_hier',
    'Hier·return': 'experiments/results/daily_btc_hier_return',
}
METRICS = ['mape', 'directional_accuracy', 'sharpe_ratio',
           'directional_accuracy_3class', 'sharpe_ratio_3class']

def latest_json(d):
    js = [f for f in glob.glob(f'{d}/*.json') if 'intermediate' not in f]
    return max(js, key=os.path.getmtime) if js else None

agg_rows, sig_rows = [], []
for exp_name, d in RESULT_DIRS.items():
    jf = latest_json(d)
    if not jf:
        print(f'[skip] {exp_name}: no results'); continue
    res = json.load(open(jf))
    for variant, vdata in res['model_results'].items():
        agg = vdata.get('aggregated', {})
        row = {'experiment': exp_name, 'variant': variant}
        for m in METRICS:
            row[m] = agg.get(m, {}).get('mean', np.nan)
        agg_rows.append(row)
        tm = vdata['individual_runs'][0]['test_metrics']
        if tm.get('predictions'):
            pr = np.array(tm['predictions'], float) / np.array(tm['prev_closes'], float) - 1
            tr = np.array(tm['targets'], float) / np.array(tm['prev_closes'], float) - 1
            corr = np.corrcoef(pr, tr)[0, 1] if pr.std() and tr.std() else np.nan
            da = (np.sign(pr) == np.sign(tr)).mean() * 100
            sig_rows.append({'experiment': exp_name, 'variant': variant,
                             'corr': round(corr, 4), 'dir_agree_%': round(da, 1)})

agg_df = pd.DataFrame(agg_rows).round(3)
sig_df = pd.DataFrame(sig_rows)
pd.set_option('display.width', 220, 'display.max_rows', 120)
print('=== Metrics (mean over 3 seeds) ==='); display(agg_df)
print('=== Signal quality (corr, dir-agree) ==='); display(sig_df)

## 4. Per-experiment comparison figures

In [ ]:
from IPython.display import Image, display
for exp_name, d in RESULT_DIRS.items():
    jf = latest_json(d)
    if not jf:
        continue
    out = f'{d}/figures'
    print(f'\n=== {exp_name} ===')
    !python experiments/visualize_results.py --results "{jf}" --output "{out}" 2>&1 | tail -1
    for fig in ['metric_comparison.png', 'box_plots.png', 'radar_chart.png']:
        p = f'{out}/{fig}'
        if os.path.exists(p):
            display(Image(filename=p, width=820))

## 5. Backtest every return-mode variant

Daily bars → `--freq 1d`, next-bar-open execution, fees+slippage. Renders backtest-comparison bars, signal-quality scatter grid, and equity overlay vs buy & hold.

In [ ]:
RETURN_DIRS = [
    'experiments/results/daily_btc_ohlc_return',
    'experiments/results/daily_btc_hier_return',
]
for d in RETURN_DIRS:
    if not glob.glob(f'{d}/*_seed42_predictions.csv'):
        print(f'[skip] {d}: no seed-42 predictions'); continue
    print(f'\n{"="*70}\nBACKTEST ALL: {os.path.basename(d)}\n{"="*70}')
    !.venv-backtest/bin/python scripts/backtest_all.py \
        --results-dir "{d}" \
        --ohlc data/cache/lunarcrush_btc_day_full.csv \
        --seed 42 --freq 1d --fees 0.001 --slippage 0.0005

In [ ]:
for d in RETURN_DIRS:
    bt = f'{d}/backtest_all'
    if not os.path.isdir(bt):
        continue
    print(f'\n=== {os.path.basename(d)} ===')
    for png in sorted(glob.glob(f'{bt}/*.png')):
        print(png); display(Image(filename=png, width=900))

## 6. Save everything to Google Drive

In [ ]:
import shutil
from datetime import datetime
GDRIVE = '/content/drive/MyDrive/thesis_results_daily_backtest'
run_dir = f"{GDRIVE}/{datetime.now().strftime('%Y%m%d_%H%M%S')}"
os.makedirs(run_dir, exist_ok=True)
for exp_name, d in RESULT_DIRS.items():
    if os.path.exists(d):
        shutil.copytree(d, f'{run_dir}/{os.path.basename(d)}', dirs_exist_ok=True)
agg_df.to_csv(f'{run_dir}/metrics_table.csv', index=False)
sig_df.to_csv(f'{run_dir}/signal_quality_table.csv', index=False)
print(f'Saved all results + tables to: {run_dir}')

## Notes

- **Why daily?** Daily returns are less noise-dominated than 4h, so if any architecture has real predictive signal, the `corr` / `dir-agree` in the signal-quality table should be visibly higher here than in the 4h notebook.
- The legacy `test_metrics.sharpe_ratio` is the toy sign-based Sharpe. The real fee-aware Sharpe is in each `backtest_all/*_summary.csv`.
- A non-trading variant shows `sharpe=n/a` — expected.
- To try the directional loss, add `loss_type: directional_mse` + `lambda_dir: 0.1` to a return-mode base config's `training:` block.